# AI作曲アレンジ（v1.0.3）

好きな音源に伴奏やドラムを加えてみましょう。音色や音量を変えると、同じメロディでも雰囲気が変わります。

2026年度3Q（月3木3クラス）の学生2名による作品をもとに、公開用に整理・改良しました。

## 最初の操作

1. **セル1 → セル2** の左の ▶ を順に押します。CPUのままで使えます。
2. **セル3** でYouTubeのURLを貼るか、デモの曲を選んで ▶ を押します。
3. **セル4** の ▶ を押します。原曲・追加パート・ミックスの再生欄が表示されます。
4. 保存したい場合は、**セル5** で形式を選んで ▶ を押します。

セル3は、URLがあればYouTube、空欄ならデモを使います。設定を変えた後は、そのセルをもう一度実行してください。

コードは折りたたんであります。読みたいときはセルのメニューから「コードを表示」を選べます。

初回の準備や解析には数分かかる場合があります。実行マークが止まるまで待ってください。
[操作マニュアル](https://github.com/shogo-ishikawa/2026_3Q_M3Th3/blob/main/projects/ai-music-arranger/docs/manual.md)
／ [コードと改良内容](https://github.com/shogo-ishikawa/2026_3Q_M3Th3/tree/main/projects/ai-music-arranger)

音源は、自分で制作したものや利用が認められたものを使ってください。

## 1. 環境を準備する

▶ を押して準備が終わるまで待ちます。初回は数分かかることがあります。
APIキーやGPUは不要です。

In [ ]:
#@title 1. 環境を準備する { display-mode: "form" }
"""Colab の初回準備。ノートブックにも同じコードを収録します。"""

from pathlib import Path
import hashlib
import importlib
import os
import platform
import re
import shutil
import subprocess
import sys
import tempfile
import urllib.request
import zipfile


PACKAGES = [
    "librosa>=0.11,<0.12", "soundfile>=0.13,<0.14",
    "pretty_midi>=0.2.10,<0.3", "ipywidgets>=8.1,<9",
    "matplotlib>=3.9,<4",
]
DENO_VERSION = "v2.9.5"


def supported_js_runtime():
    """yt-dlp が対応する JavaScript 実行環境があるか調べます。"""
    for name, minimum in [("deno", (2, 3, 0)), ("node", (22, 0, 0))]:
        executable = shutil.which(name)
        if not executable:
            continue
        try:
            result = subprocess.run(
                [executable, "--version"], capture_output=True,
                text=True, timeout=10, check=True,
            )
            match = re.search(r"(\d+)\.(\d+)\.(\d+)", result.stdout)
            if match and tuple(map(int, match.groups())) >= minimum:
                return name
        except (OSError, subprocess.SubprocessError):
            continue
    return None


def install_deno(bin_dir):
    """公式配布の Deno をチェックサムで照合して配置します（Linux Colab 用）。"""
    if platform.system() != "Linux" or platform.machine() not in {"x86_64", "AMD64"}:
        raise RuntimeError("この環境では Deno または Node.js を手動で導入してください。")
    filename = "deno-x86_64-unknown-linux-gnu.zip"
    base = f"https://github.com/denoland/deno/releases/download/{DENO_VERSION}/"
    with urllib.request.urlopen(base + filename + ".sha256sum", timeout=30) as response:
        expected = response.read(4096).decode("utf-8").split()[0]
    if not re.fullmatch(r"[a-fA-F0-9]{64}", expected):
        raise RuntimeError("Deno のチェックサムを確認できませんでした。")
    bin_dir = Path(bin_dir)
    bin_dir.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(dir=bin_dir) as temporary:
        archive = Path(temporary) / filename
        size = 0
        with urllib.request.urlopen(base + filename, timeout=60) as response, archive.open("wb") as out:
            while chunk := response.read(1024 * 1024):
                size += len(chunk)
                if size > 100 * 1024 * 1024:
                    raise RuntimeError("Deno の取得サイズが想定を超えました。")
                out.write(chunk)
        if hashlib.sha256(archive.read_bytes()).hexdigest() != expected.lower():
            raise RuntimeError("Deno の照合に失敗しました。時間をおいて準備セルを再実行してください。")
        with zipfile.ZipFile(archive) as zipped:
            executable = Path(temporary) / "deno"
            executable.write_bytes(zipped.read("deno"))
        executable.chmod(0o755)
        executable.replace(bin_dir / "deno")
    os.environ["PATH"] = str(bin_dir.resolve()) + os.pathsep + os.environ.get("PATH", "")


def setup_environment():
    """準備が済んだら、セル 2 のプログラムを保存する場所へ移動します。"""
    try:
        import google.colab  # noqa: F401
        in_colab = True
    except ImportError:
        in_colab = False
    base = Path("/content") if in_colab else Path.cwd()
    runtime_dir = base if base.name == ".ai_music_arranger_runtime" else base / ".ai_music_arranger_runtime"
    runtime_dir.mkdir(parents=True, exist_ok=True)
    print("1/3 音声解析と操作画面に必要なライブラリを準備しています。")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES],
        check=True, timeout=360,
    )
    # YouTubeの仕様変更に備え、取得ツールと対応するEJSを一緒に更新します。
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "yt-dlp[default]"],
        check=True, timeout=180,
    )
    importlib.invalidate_caches()

    print("2/3 音声を変換する FFmpeg を確認しています。")
    if not shutil.which("ffmpeg"):
        if not in_colab:
            raise RuntimeError("FFmpeg がありません。README のローカル実行手順を確認してください。")
        subprocess.run(["apt-get", "-qq", "update"], check=True, timeout=120)
        subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True, timeout=180)

    bin_dir = runtime_dir / "bin"
    if bin_dir.exists():
        os.environ["PATH"] = str(bin_dir.resolve()) + os.pathsep + os.environ.get("PATH", "")
    print("3/3 YouTube 取得に必要な JavaScript 実行環境を確認しています。")
    if not supported_js_runtime():
        try:
            install_deno(bin_dir)
        except Exception:
            print("YouTube 用の追加準備が完了しませんでした。準備セルを再実行してください。")
            print("デモ音源と PC からのファイル読み込みはそのまま試せます。")
    runtime = supported_js_runtime()
    print(f"YouTube 用実行環境: {runtime or '未準備（操作マニュアルの対処方法を参照）'}")

    # 操作はColab標準フォームなので、追加のウィジェット描画設定は不要です。
    os.chdir(runtime_dir)
    if str(runtime_dir.resolve()) not in sys.path:
        sys.path.insert(0, str(runtime_dir.resolve()))
    print("準備が完了しました。続けてセル 2 の各プログラムと、セル 3 を実行してください。")


if __name__ == "__main__":
    setup_environment()


## 2. プログラムを読み込む

▶ を1回押すと、音源の取り込み・解析・再生に使うプログラムをまとめて読み込みます。
準備ができたらセル3へ進みましょう。

In [ ]:
#@title 2. プログラムを読み込む { display-mode: "form" }
from pathlib import Path
import importlib
import importlib.util
import sys

module_sources = {
    'audio_input.py': r'''"""音源の取り込み。YouTube取得と音声の切り出しを音楽処理から分けます。"""

from pathlib import Path
import contextlib
import math
import os
import re
import shutil
import signal
import subprocess
import sys
import tempfile
from urllib.parse import parse_qs, urlsplit

import numpy as np
import soundfile as sf


SAMPLE_RATE = 22050
MIN_DURATION = 3.0
MAX_DURATION = 60.0
DOWNLOAD_TIMEOUT = 180
DECODE_TIMEOUT = 60


class AudioInputError(ValueError):
    """利用者が次の操作を判断できる、短い取り込みエラー。"""


def _run_process(command, timeout, text=False):
    """Colabでは子のFFmpegも同じグループに置き、時間切れ時に止めます。"""
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                               text=text, start_new_session=(os.name == "posix"))
    try:
        stdout, stderr = process.communicate(timeout=timeout)
    except BaseException:
        # yt-dlpだけを止めると、その子のFFmpegが通信を続ける場合があります。
        with contextlib.suppress(ProcessLookupError):
            if os.name == "posix":
                os.killpg(process.pid, signal.SIGKILL)
            else:
                process.kill()
        process.communicate()
        raise
    return subprocess.CompletedProcess(command, process.returncode, stdout, stderr)


def normalise_youtube_url(url):
    """通常の動画・短縮URL・Shorts・ライブのURLを単一動画URLに直します。"""
    if not isinstance(url, str):
        raise AudioInputError("YouTubeの動画URLを入力してください。")
    value = url.strip()
    try:
        parts = urlsplit(value)
        # 不正なポート番号もここで検出します。
        port = parts.port
    except ValueError:
        raise AudioInputError("YouTubeの動画URLを確認してください。") from None
    if (parts.scheme != "https" or parts.username or parts.password
            or port not in (None, 443)):
        raise AudioInputError("https:// で始まるYouTubeの動画URLを入力してください。")
    host = (parts.hostname or "").lower()
    if host == "youtu.be":
        video_id = parts.path.strip("/")
    elif host in {"youtube.com", "www.youtube.com", "m.youtube.com", "music.youtube.com"}:
        if parts.path == "/watch":
            ids = parse_qs(parts.query).get("v", [])
            video_id = ids[0] if len(ids) == 1 else ""
        else:
            match = re.fullmatch(r"/(?:shorts|live|embed)/([A-Za-z0-9_-]{11})/?", parts.path)
            video_id = match.group(1) if match else ""
    else:
        raise AudioInputError("YouTubeまたはyoutu.beの動画URLを入力してください。")
    if not re.fullmatch(r"[A-Za-z0-9_-]{11}", video_id):
        raise AudioInputError("単一動画のURLを入力してください。チャンネルや再生リストは使えません。")
    return "https://www.youtube.com/watch?v=" + video_id


def _validate_section(start, duration):
    try:
        start, duration = float(start), float(duration)
    except (ValueError, TypeError):
        raise AudioInputError("開始位置と長さは秒数で入力してください。") from None
    if not math.isfinite(start) or start < 0:
        raise AudioInputError("開始位置は0秒以上にしてください。")
    if not math.isfinite(duration) or not MIN_DURATION <= duration <= MAX_DURATION:
        raise AudioInputError("取り込む長さは3〜60秒にしてください。")
    return start, duration


def find_js_runtime():
    """yt-dlpが対応するDenoまたはNodeを探します。見つからなければNone。"""
    for name, minimum in (("deno", (2, 3)), ("node", (22, 0))):
        path = shutil.which(name)
        if not path:
            continue
        try:
            result = subprocess.run([path, "--version"], capture_output=True,
                                    text=True, timeout=10, check=False)
            match = re.search(r"(?:^|\s|v)(\d+)\.(\d+)", result.stdout)
            if result.returncode == 0 and match:
                version = tuple(int(x) for x in match.groups())
                if version >= minimum:
                    return name, path
        except (OSError, subprocess.TimeoutExpired):
            continue
    return None


def load_audio_file(path, start=0.0, duration=20.0, sr=SAMPLE_RATE):
    """FFmpegで指定区間だけを読み、モノラルのfloat32配列を返します。

    取り込む長さは3〜60秒。曲末まではそれより短くても読み込めます。
    ファイル全体をメモリに展開せず、変換は最大60秒で打ち切ります。
    """
    start, duration = _validate_section(start, duration)
    if isinstance(sr, bool) or not isinstance(sr, int) or not 8000 <= sr <= 48000:
        raise AudioInputError("サンプリング周波数は8000〜48000 Hzの整数にしてください。")
    source = Path(path)
    if not source.is_file():
        raise AudioInputError("音声ファイルが見つかりません。ファイルを選び直してください。")
    ffmpeg = shutil.which("ffmpeg")
    if not ffmpeg:
        raise AudioInputError("FFmpegがありません。先に環境セットアップのセルを実行してください。")
    command = [ffmpeg, "-nostdin", "-hide_banner", "-loglevel", "error",
               "-ss", f"{start:.6f}", "-i", str(source.resolve()),
               "-t", f"{duration:.6f}", "-vn", "-ac", "1", "-ar", str(sr),
               "-f", "f32le", "pipe:1"]
    try:
        result = _run_process(command, timeout=DECODE_TIMEOUT)
    except subprocess.TimeoutExpired:
        raise AudioInputError("音声の変換が60秒以内に終わりませんでした。短い区間で試してください。") from None
    except OSError:
        raise AudioInputError("音声の変換を開始できません。環境セットアップをやり直してください。") from None
    if result.returncode != 0 or len(result.stdout) % 4:
        raise AudioInputError("音声を読み込めません。WAV・MP3などの音声ファイルを選んでください。")
    samples = np.frombuffer(result.stdout, dtype="<f4").copy()
    # 万一デコーダーが長めに返しても、指定区間のサンプル数に制限します。
    samples = samples[:int(round(duration * sr))]
    if samples.size == 0 or not np.all(np.isfinite(samples)):
        raise AudioInputError("指定した区間に読み込める音声がありません。開始位置を確認してください。")
    return samples


def _youtube_error_message(stderr):
    """診断文は表示せず、分かった原因だけを短い案内に置き換えます。"""
    if isinstance(stderr, bytes):
        stderr = stderr.decode("utf-8", errors="replace")
    detail = str(stderr or "").casefold()
    if any(text in detail for text in (
            "no module named yt_dlp", "no module named 'yt_dlp'",
            "no supported javascript runtime", "ffmpeg not found",
            "ffprobe or ffmpeg not found")):
        return "取得ツールの準備ができていません。セル1を実行してから、もう一度試してください。"
    if any(text in detail for text in (
            "private video", "video is private", "video unavailable", "this video is unavailable",
            "video has been removed", "not available in your country",
            "not available in your region", "not made this video available")):
        return "この動画は公開状態や地域などの制限で取得できません。別の動画で試してください。"
    if any(text in detail for text in (
            "sign in", "log in", "login required", "authentication required",
            "not a bot", "confirm your age", "age-restricted", "http error 401", "http 401")):
        return "YouTubeからログインや追加の確認を求められました。別の音源で試してください。"
    if any(text in detail for text in (
            "http error 429", "http 429", "too many requests", "rate limit")):
        return "取得が一時的に制限されています。少し時間をおいて、セル3を実行してください。"
    if any(text in detail for text in (
            "http error 403", "http 403", "403 forbidden", "access denied",
            "request blocked")):
        return "音声の配信先から接続を拒否されました。時間をおいて試すか、別の音源を選んでください。"
    if any(text in detail for text in (
            "certificate_verify_failed", "certificate verify failed",
            "ssl error", "ssl:", "tls handshake")):
        return "YouTubeへの安全な接続を確認できませんでした。接続環境を確認してから、もう一度試してください。"
    if any(text in detail for text in (
            "timed out", "timeout", "name or service not known",
            "temporary failure in name resolution", "unable to resolve",
            "network is unreachable", "connection refused", "connection reset",
            "remote end closed connection")):
        return "YouTubeへ接続できませんでした。接続を確認し、もう一度試してください。"
    if any(text in detail for text in (
            "postprocessing:", "post-processing:", "ffmpeg exited",
            "ffmpeg returned", "invalid data found when processing input",
            "error opening input", "audio conversion failed")):
        return "取得したデータを音声に変換できませんでした。別の動画や短い区間で試してください。"
    return "YouTubeから音声を取得できませんでした。URLと動画の公開状態を確認し、もう一度試してください。"


def download_youtube(url, destination, start=0.0, duration=20.0):
    """指定区間をYouTubeから取得し、22050 Hz・モノラルWAVとして保存。

    yt-dlpには指定した区間だけを要求します。配信方式によって通信量が
    区間長に比例するとは限りません。取得は最大180秒、続く変換は最大60秒。
    失敗時には、前回成功したdestinationを変更しません。
    """
    canonical_url = normalise_youtube_url(url)
    start, duration = _validate_section(start, duration)
    if not shutil.which("ffmpeg"):
        raise AudioInputError("FFmpegがありません。先に環境セットアップのセルを実行してください。")
    runtime = find_js_runtime()
    if runtime is None:
        raise AudioInputError("YouTube取得にはDeno 2.3以上またはNode 22以上が必要です。環境セットアップをやり直してください。")
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    name, executable = runtime
    with tempfile.TemporaryDirectory(prefix="youtube-", dir=destination.parent) as staging:
        staging = Path(staging)
        template = staging / "source.%(ext)s"
        command = [sys.executable, "-m", "yt_dlp", "--ignore-config", "--no-playlist",
                   "--no-progress", "--no-warnings", "--socket-timeout", "20",
                   "--retries", "2", "--fragment-retries", "2",
                   "--js-runtimes", f"{name}:{executable}", "-f", "bestaudio/best",
                   "--download-sections", f"*{start:.6f}-{start + duration:.6f}",
                   "-x", "--audio-format", "wav", "-o", str(template)]
        if os.environ.get("SSL_CERT_FILE"):
            # 明示的に設定されたCAを尊重します。TLSの証明書・ホスト名検証は継続。
            command += ["--compat-options", "no-certifi"]
        command += ["--", canonical_url]
        try:
            result = _run_process(command, text=True, timeout=DOWNLOAD_TIMEOUT)
        except subprocess.TimeoutExpired:
            raise AudioInputError("YouTubeからの取得が180秒以内に終わりませんでした。短い区間で再試行してください。") from None
        except OSError:
            raise AudioInputError("YouTube取得を開始できません。環境セットアップをやり直してください。") from None
        source = staging / "source.wav"
        if result.returncode != 0 or not source.is_file():
            raise AudioInputError(_youtube_error_message(result.stderr))
        try:
            samples = load_audio_file(source, start=0.0, duration=duration)
        except AudioInputError:
            raise AudioInputError("取得した音声を読み込めませんでした。短い区間や別の動画で試してください。") from None
        converted = staging / "validated.wav"
        try:
            sf.write(converted, samples, SAMPLE_RATE, subtype="PCM_16")
            os.replace(converted, destination)
        except (OSError, RuntimeError):
            raise AudioInputError("取得した音源を保存できません。保存先の空き容量を確認してください。") from None
    return destination
''',
    'music_core.py': r'''"""音源から伴奏を作る、画面表示と分離した処理。

2026年度3Qの学生作品にあったpYIN・コード伴奏・ドラムを引き継ぐ。
音声の分離モデルや大規模な生成モデルは使わず、推定と規則で編曲する。
"""
from dataclasses import dataclass
import math

import librosa
import numpy as np
import pretty_midi

SAMPLE_RATE = 22050
NOTE_NAMES = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
SCALE_DEFS = {
    "major": [0, 2, 4, 5, 7, 9, 11],
    "natural_minor": [0, 2, 3, 5, 7, 8, 10],
    "harmonic_minor": [0, 2, 3, 5, 7, 8, 11],
    "dorian": [0, 2, 3, 5, 7, 9, 10],
    "mixolydian": [0, 2, 4, 5, 7, 9, 10],
    "pentatonic": [0, 2, 4, 7, 9],
}
INSTRUMENTS = {"Electric Piano": 4, "Acoustic Grand Piano": 0,
               "Acoustic Guitar": 24, "Electric Guitar": 27,
               "Strings": 48, "Synth Lead": 80, "Music Box": 10}

# 音の高さはMIDI番号、長さは四分音符を1とした拍数です。
# 有名曲も既存の録音は使わず、保護期間の終了した旋律をここで演奏します。
DEMO_CATALOGUE = {
    "オリジナル（ポップ）": {
        "title": "オリジナル（ポップ）", "bpm": 120, "key": "C", "scale": "major",
        "description": "短い音が続く、明るい練習用メロディです。",
        "duration_seconds": 12.0,
        "source": "この教材で作成したオリジナルの旋律。",
        "copyright": "教材用に作成した旋律をプログラムで合成。外部の録音は使用していません。",
        "pitches": (60, 64, 67, 72, 65, 64, 62, 60, 60, 64, 65, 69, 67, 65, 62, 60),
        "beats": (1,) * 16,
    },
    "オリジナル（ゆったり）": {
        "title": "オリジナル（ゆったり）", "bpm": 80, "key": "C", "scale": "major",
        "description": "長く伸ばす音を入れた、ゆったりしたメロディです。",
        "duration_seconds": 12.0,
        "source": "この教材で作成したオリジナルの旋律。",
        "copyright": "教材用に作成した旋律をプログラムで合成。外部の録音は使用していません。",
        "pitches": (60, 64, 67, 64, 65, 69, 67, 65, 64, 62, 60),
        "beats": (1, 1, 2, 1, 1, 2, 1, 1, 2, 1, 3),
    },
    "きらきら星（冒頭）": {
        "title": "きらきら星（冒頭）", "bpm": 100, "key": "C", "scale": "major",
        "description": "おなじみの旋律で、音の高さや長さを確かめられます。",
        "duration_seconds": 9.6,
        "source": "フランス民謡 Ah! vous dirai-je, maman の主題を、ハ長調の単音に簡略化。",
        "source_url": "https://www.mutopiaproject.org/cgibin/piece-info.cgi?id=2236",
        "copyright": "保護期間が終了した主題を演奏。既存の録音や歌詞は使用していません。",
        "pitches": (60, 60, 67, 67, 69, 69, 67, 65, 65, 64, 64, 62, 62, 60),
        "beats": (1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 2),
    },
    "歓喜の歌（冒頭）": {
        "title": "歓喜の歌（冒頭）", "bpm": 120, "key": "C", "scale": "major",
        "description": "ベートーヴェンの旋律です。伴奏や音色を変えて聴き比べてみましょう。",
        "duration_seconds": 16.0,
        "source": "L. van Beethoven, Symphony No. 9, Op. 125（1824年）の主題をハ長調で演奏。",
        "source_url": "https://www.mutopiaproject.org/cgibin/piece-info.cgi?id=528",
        "copyright": "保護期間が終了した主題を演奏。既存の録音や歌詞は使用していません。",
        "pitches": (64, 64, 65, 67, 67, 65, 64, 62, 60, 60, 62, 64, 64, 62, 62,
                    64, 64, 65, 67, 67, 65, 64, 62, 60, 60, 62, 64, 62, 60, 60),
        "beats": (1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1.5, .5, 2,
                  1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1.5, .5, 2),
    },
}
DEMO_NAMES = tuple(DEMO_CATALOGUE)


def get_demo_info(demo_name=DEMO_NAMES[0]):
    """画面に表示するデモの説明と、フレーズを聴ける推奨時間を返します。"""
    if demo_name not in DEMO_CATALOGUE:
        raise ValueError("デモ音源を一覧から選んでください。")
    return {name: value for name, value in DEMO_CATALOGUE[demo_name].items()
            if name not in {"pitches", "beats"}}


@dataclass(frozen=True)
class Note:
    pitch: int
    start: float
    end: float
    velocity: int = 90


@dataclass
class ArrangementConfig:
    bpm: float = 120.0
    root: str = "C"
    scale: str = "major"
    instrument: str = "Electric Piano"
    add_chords: bool = True
    add_drums: bool = True
    add_melody: bool = True
    original_gain: float = 0.7
    chord_gain: float = 0.6
    drum_gain: float = 0.8
    melody_gain: float = 0.6
    beat_offset: float = 0.0
    humanise: float = 0.0
    seed: int = 42


def _audio(y, sr):
    y = np.asarray(y, dtype=np.float64)
    if y.ndim != 1 or y.size == 0 or not np.isfinite(y).all():
        raise ValueError("音源は空でない、有限の数値からなるモノラル音声にしてください。")
    if isinstance(sr, bool) or not isinstance(sr, (int, np.integer)) or not 8000 <= sr <= 192000:
        raise ValueError("サンプリング周波数は8000〜192000 Hzの整数にしてください。")
    return y


def _validate(config):
    if config.root not in NOTE_NAMES or config.scale not in SCALE_DEFS:
        raise ValueError("キーまたはスケールが選択肢にありません。")
    if config.instrument not in INSTRUMENTS:
        raise ValueError("楽器が選択肢にありません。")
    for name, low, high in [("bpm", 40, 240), ("beat_offset", 0, 60),
                            ("humanise", 0, 1), ("original_gain", 0, 2),
                            ("chord_gain", 0, 2), ("drum_gain", 0, 2),
                            ("melody_gain", 0, 2)]:
        value = getattr(config, name)
        if not isinstance(value, (int, float, np.number)) or not math.isfinite(value) or not low <= value <= high:
            raise ValueError(f"{name}は{low}〜{high}の有限の数値にしてください。")
    if not isinstance(config.seed, (int, np.integer)) or not 0 <= config.seed < 2**32:
        raise ValueError("乱数シードは0〜4294967295の整数にしてください。")


def make_demo(duration=12.0, sr=SAMPLE_RATE, demo_name=DEMO_NAMES[0]):
    """選んだ旋律を演奏する。指定時間を超える場合は切り、短ければ繰り返す。"""
    if not isinstance(duration, (int, float)) or not math.isfinite(duration) or not 0.2 <= duration <= 60:
        raise ValueError("デモの長さは0.2〜60秒にしてください。")
    get_demo_info(demo_name)
    _audio(np.zeros(1), sr)
    y = np.zeros(round(duration * sr))
    demo = DEMO_CATALOGUE[demo_name]
    beat_seconds = 60 / demo["bpm"]
    start, index = 0.0, 0
    while start < duration:
        note_index = index % len(demo["pitches"])
        note_length = beat_seconds * demo["beats"][note_index]
        left, right = round(start * sr), min(len(y), round((start + note_length * .9) * sr))
        t = np.arange(right - left) / sr
        freq = librosa.midi_to_hz(demo["pitches"][note_index])
        envelope = np.minimum(t / 0.012, 1) * np.minimum((len(t) / sr - t) / 0.035, 1)
        y[left:right] += 0.4 * envelope * np.exp(-1.5 * t) * np.sin(2 * np.pi * freq * t)
        start += note_length
        index += 1
    return y.astype(np.float32)


def analyse_audio(y, sr=SAMPLE_RATE):
    """BPM・拍の位相・長調/短調を推定する。confidenceは比較用の目安。"""
    y = _audio(y, sr)
    result = dict(bpm=120.0, key="C", scale="major", beat_offset=0.0,
                  confidence=0.0, note="推定値は目安です。原曲を聴きながら修正してください。")
    if len(y) < sr * 0.12 or np.max(np.abs(y)) < 1e-6:
        result["note"] = "音源が短いか無音のため、仮の設定（C長調・120 BPM）を表示しています。"
        return result
    tempo, beats = librosa.beat.beat_track(y=y, sr=sr, trim=False)
    bpm = float(np.asarray(tempo).reshape(-1)[0])
    if math.isfinite(bpm) and bpm > 0:
        while bpm < 60:
            bpm *= 2
        while bpm > 200:
            bpm /= 2
        result["bpm"] = round(bpm, 2)
        if len(beats):
            first = float(librosa.frames_to_time(beats[0], sr=sr))
            result["beat_offset"] = round(first % (60 / bpm), 3)
    # 一番大きな音を主音とせず、12音全体と調のプロファイルを比較する。
    padded = np.pad(y, (0, max(0, 4096 - len(y))))
    chroma = librosa.feature.chroma_stft(y=padded, sr=sr, n_fft=4096).mean(axis=1)
    profiles = {
        "major": np.array([6.35, 2.23, 3.48, 2.33, 4.38, 4.09, 2.52, 5.19, 2.39, 3.66, 2.29, 2.88]),
        "natural_minor": np.array([6.33, 2.68, 3.52, 5.38, 2.60, 3.53, 2.54, 4.75, 3.98, 2.69, 3.34, 3.17]),
    }
    scores = []
    if np.std(chroma) > 1e-8:
        for mode, profile in profiles.items():
            for root in range(12):
                score = float(np.corrcoef(chroma, np.roll(profile, root))[0, 1])
                scores.append((score, root, mode))
        scores.sort(reverse=True)
        best, root, mode = scores[0]
        result.update(key=NOTE_NAMES[root], scale=mode,
                      confidence=round(max(0.0, best - scores[1][0]), 3))
    return result


def pitches_to_notes(f0, voiced, sr, hop_length, duration, min_duration=0.08):
    """ピッチの連続区間を音符にする。最後に続いている音も確定する。"""
    notes = []
    current, start = None, 0.0
    for index, (freq, is_voiced) in enumerate(zip(f0, voiced)):
        t = min(index * hop_length / sr, duration)
        pitch = int(round(librosa.hz_to_midi(freq))) if is_voiced and np.isfinite(freq) and freq > 0 else None
        if pitch is not None and not 0 <= pitch <= 127:
            pitch = None
        if pitch != current:
            if current is not None and t - start >= min_duration:
                notes.append(Note(current, start, t))
            current, start = pitch, t
    if current is not None and duration - start >= min_duration:
        notes.append(Note(current, start, duration))
    return notes


def extract_melody(y, sr=SAMPLE_RATE):
    """pYINで単音の高さを推定。多重奏の主旋律を分離する処理ではない。"""
    y = _audio(y, sr)
    if len(y) < sr * 0.08 or np.max(np.abs(y)) < 1e-6:
        return []
    frame_length, hop_length = 2048, 512
    padded = np.pad(y, (0, max(0, frame_length - len(y))))
    f0, voiced, _ = librosa.pyin(padded, fmin=librosa.note_to_hz("C3"),
                                fmax=min(librosa.note_to_hz("C6"), sr / 2 - 1),
                                sr=sr, frame_length=frame_length, hop_length=hop_length,
                                n_thresholds=50, resolution=0.2)
    return pitches_to_notes(f0, voiced, sr, hop_length, len(y) / sr)


def chord_pitches(root, scale, degree):
    """スケール上の音を1音おきに3つ重ねる。degreeは0から数える。"""
    intervals = SCALE_DEFS[scale]
    tonic = 48 + NOTE_NAMES.index(root)
    pitches = []
    for step in (0, 2, 4):
        octave, index = divmod(degree + step, len(intervals))
        pitches.append(tonic + 12 * octave + intervals[index])
    return pitches


def _scaled_note(pitch, start, end, velocity, gain, duration, rng, humanise):
    if gain <= 0 or velocity <= 0:
        return None
    start = max(0, start + rng.uniform(-0.018, 0.018) * humanise)
    end = min(duration, end)
    velocity = int(np.clip(round((velocity + rng.uniform(-10, 10) * humanise) * gain), 1, 127))
    if start >= end:
        return None
    return Note(int(pitch), float(start), float(end), velocity)


def _render(notes, sr, sample_count, program=4, drums=False, seed=42):
    """サンプル音源なしで聴ける簡易合成。GM音源の忠実な再現ではない。"""
    audio = np.zeros(sample_count)
    rng = np.random.default_rng(seed)
    for note in notes:
        left, right = round(note.start * sr), min(sample_count, round(note.end * sr))
        if right <= left:
            continue
        t = np.arange(right - left) / sr
        if drums:
            if note.pitch == 36:
                wave = np.sin(2 * np.pi * (48 * t + 24 * (1 - np.exp(-30 * t)) / 30)) * np.exp(-22 * t)
            else:
                noise = rng.uniform(-1, 1, len(t))
                wave = noise * np.exp(-(65 if note.pitch == 42 else 28) * t)
            amplitude = 0.28
        else:
            frequency = librosa.midi_to_hz(note.pitch)
            # 倍音を変えることで音色の違いを作る。高域の折り返しを避ける。
            weights = {0: [1, .22, .10], 4: [1, .35, .08], 24: [1, .30, .18],
                       27: [1, .45, .23], 48: [1, .55, .27], 80: [1, .45, .30],
                       10: [1, .05, .30]}[program]
            wave = np.zeros(len(t))
            for harmonic, weight in enumerate(weights, 1):
                if harmonic * frequency < sr / 2:
                    wave += weight * np.sin(2 * np.pi * frequency * harmonic * t)
            decay = 0.6 if program in (48, 80) else 3.0
            wave *= np.exp(-decay * t)
            amplitude = 0.16
        attack = np.minimum(t / 0.008, 1)
        release = np.minimum((len(t) / sr - t) / 0.025, 1)
        audio[left:right] += amplitude * (note.velocity / 100) * wave * attack * release
    return audio


def arrange(y, sr, config, melody_notes=None):
    """伴奏の拍を指定BPMで配置し、元の音源と同じ長さにミックスする。

    元の音声の速度や音程は変えない。BPM変更は追加する伴奏に作用する。
    """
    y = _audio(y, sr)
    _validate(config)
    duration = len(y) / sr
    rng = np.random.default_rng(config.seed)
    tracks = {"chords": [], "drums": [], "melody": []}
    warnings = []
    beat = 60 / config.bpm
    bar = 4 * beat
    # 7音のスケールは I–V–vi–IV。5音ではスケール上の対応音を使う。
    progression = [0, 4, 5, 3] if len(SCALE_DEFS[config.scale]) == 7 else [0, 3, 4, 2]
    for step, t in enumerate(np.arange(config.beat_offset, duration, beat / 4)):
        step_in_bar = step % 16
        if config.add_chords and step_in_bar in [0, 3, 6, 8, 10, 12, 14]:
            degree = progression[(step // 16) % 4]
            for index, pitch in enumerate(chord_pitches(config.root, config.scale, degree)):
                start = t + index * 0.012
                note = _scaled_note(pitch, start, start + min(.25, beat * .6), 85,
                                    config.chord_gain, duration, rng, config.humanise)
                if note:
                    tracks["chords"].append(note)
        if config.add_drums:
            for pitch, pattern, velocity, length in [(36, [0, 10, 14], 110, .15),
                                                      (38, [4, 12], 100, .15),
                                                      (42, list(range(0, 16, 2)), 75, .05)]:
                if step_in_bar in pattern:
                    note = _scaled_note(pitch, t, t + length, velocity,
                                        config.drum_gain, duration, rng, config.humanise)
                    if note:
                        tracks["drums"].append(note)
    if config.add_melody and config.melody_gain > 0:
        source_notes = extract_melody(y, sr) if melody_notes is None else melody_notes
        for source in source_notes:
            if (not 0 <= source.pitch <= 127 or not math.isfinite(source.start)
                    or not math.isfinite(source.end) or source.start < 0
                    or source.end <= source.start or not 0 <= source.velocity <= 127):
                raise ValueError("メロディの音符データに不正な値があります。")
            note = _scaled_note(source.pitch, source.start, source.end, source.velocity,
                                config.melody_gain, duration, rng, 0)
            if note:
                tracks["melody"].append(note)
        if not tracks["melody"]:
            warnings.append("メロディの音符が見つかりませんでした。単音がはっきりした区間を試してください。")
    midi = pretty_midi.PrettyMIDI(initial_tempo=config.bpm)
    accompaniment = np.zeros(len(y))
    for name, notes in tracks.items():
        program = INSTRUMENTS[config.instrument] if name == "chords" else (0 if name == "drums" else 80)
        instrument = pretty_midi.Instrument(program, is_drum=name == "drums", name=name)
        for note in notes:
            instrument.notes.append(pretty_midi.Note(note.velocity, note.pitch, note.start, note.end))
        if notes:
            midi.instruments.append(instrument)
        accompaniment += _render(notes, sr, len(y), program, name == "drums", config.seed)
    mixed = y * config.original_gain + accompaniment
    peak = np.max(np.abs(mixed))
    if peak > 1:
        mixed /= peak
        accompaniment /= peak
        warnings.append("音割れを防ぐため、ミックス全体の音量を調整しました。")
    return dict(mixed=mixed.astype(np.float32), accompaniment=accompaniment.astype(np.float32),
                midi=midi, notes=tracks, warnings=warnings)
''',
    'notebook_session.py': r'''"""Colab標準フォームから呼ぶ処理。ウィジェットの描画を必要としません。"""
from dataclasses import asdict
import base64
from html import escape
import json
from pathlib import Path
import tempfile
import zipfile

import numpy as np
import soundfile as sf
from IPython.display import Audio, HTML, display

from audio_input import AudioInputError, download_youtube, load_audio_file
from music_core import (
    SAMPLE_RATE, ArrangementConfig, analyse_audio, arrange, extract_melody, make_demo,
    DEMO_NAMES, get_demo_info,
)

VERSION = "1.0.3"
MAX_UPLOAD_BYTES = 50 * 1024 * 1024


class NotebookSession:
    """セル3で音源を用意し、セル4で生成、セル5で保存します。"""

    def __init__(self, work_dir=None):
        self.work_dir = Path(work_dir) if work_dir is not None else Path.cwd() / "music_work"
        self.work_dir.mkdir(parents=True, exist_ok=True)
        self.state = dict(audio=None, analysis=None, melody_notes=None, result=None,
                          artifacts={}, source_revision=0, source=None, config=None)

    def _invalidate_result(self):
        self.state.update(result=None, artifacts={}, config=None)

    @staticmethod
    def _read_uploaded(uploaded, temporary, start, duration):
        if not uploaded:
            raise AudioInputError("音源が選択されていません。セル3を実行し直して、ファイルを選んでください。")
        if len(uploaded) != 1:
            raise AudioInputError("音声ファイルを1つだけ選んでください。")
        filename, content = next(iter(uploaded.items()))
        suffix = Path(filename).suffix.lower()
        if suffix not in {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".aac", ".webm"}:
            raise AudioInputError("WAV・MP3などの音声ファイルを選んでください。")
        if not content or len(content) > MAX_UPLOAD_BYTES:
            raise AudioInputError("音源は50 MiB以下の空でないファイルを選んでください。")
        path = Path(temporary) / ("uploaded" + suffix)
        path.write_bytes(content)
        return load_audio_file(path, start=start, duration=duration, sr=SAMPLE_RATE)

    def prepare_source(self, kind="URLがあればYouTube", youtube_url="", start=0, duration=20,
                       uploaded=None, demo_song=DEMO_NAMES[0]):
        """取得と解析が両方成功した場合にだけ、現在の音源を置き換えます。"""
        self._invalidate_result()
        print("音源を用意しています。少しお待ちください。")
        try:
            if kind == "URLがあればYouTube":
                if not isinstance(youtube_url, str):
                    raise AudioInputError("YouTubeのURLを文字で入力してください。")
                kind = "YouTube" if youtube_url.strip() else "デモ音源"
            demo_info = None
            if kind == "デモ音源":
                demo_info = get_demo_info(demo_song)
                if isinstance(youtube_url, str) and youtube_url.strip():
                    print("「デモ音源」が選ばれています。YouTubeを使うときは音源の選び方を変更してください。")
                print(f"今回のデモ：{demo_info['title']}")
                audio = make_demo(duration=demo_info["duration_seconds"], sr=SAMPLE_RATE,
                                  demo_name=demo_song)
                source_start = 0
            elif kind == "YouTube":
                print(f"YouTubeの指定区間（{start}秒から{duration}秒分）を取得します。")
                with tempfile.TemporaryDirectory(dir=self.work_dir) as temporary:
                    path = download_youtube(youtube_url, Path(temporary) / "input.wav",
                                            start=start, duration=duration)
                    audio, rate = sf.read(path, dtype="float32")
                    if rate != SAMPLE_RATE:
                        raise AudioInputError("音源の形式を確認できません。セル3で取得し直してください。")
                source_start = float(start)
            elif kind == "PCの音源":
                if uploaded is None:
                    try:
                        from google.colab import files
                    except ImportError:
                        raise AudioInputError("PCのファイル選択はColabで利用してください。まずデモ音源でも試せます。") from None
                    uploaded = files.upload()
                with tempfile.TemporaryDirectory(dir=self.work_dir) as temporary:
                    audio = self._read_uploaded(uploaded, temporary, start, duration)
                source_start = float(start)
            else:
                raise AudioInputError("音源の種類をデモ音源・YouTube・PCの音源から選んでください。")
            audio = np.asarray(audio, dtype=np.float32)
            if audio.ndim != 1 or len(audio) == 0 or not np.all(np.isfinite(audio)):
                raise AudioInputError("音源を読み込めません。別の音源か短い区間で試してください。")
            peak = float(np.max(np.abs(audio)))
            if peak > 1:
                audio = audio / peak
            print("テンポとキーを解析しています。初回は少し時間がかかります。")
            analysis = analyse_audio(audio, SAMPLE_RATE)
            if demo_info is not None:
                # デモは演奏条件が分かるため、伴奏も同じテンポ・調にそろえます。
                analysis.update(bpm=demo_info["bpm"], key=demo_info["key"],
                                scale=demo_info["scale"], beat_offset=0.0,
                                note="デモのテンポとキーに合わせています。")
        except (AudioInputError, ValueError) as error:
            print("準備できませんでした：" + str(error))
            if self.state["audio"] is not None:
                print("前の音源は保持しています。保存結果は無効にしたため、必要ならセル4で生成し直してください。")
            return False
        except Exception:
            print("音源の準備または解析を完了できませんでした。準備セルを実行し直し、デモ音源で試してください。")
            if self.state["audio"] is not None:
                print("前の音源は保持しています。必要ならセル4で生成し直してください。")
            return False
        self.state.update(audio=audio, analysis=analysis, melody_notes=None,
                          source_revision=self.state["source_revision"] + 1,
                          source=dict(kind=kind, start_seconds=source_start,
                                      duration_seconds=round(len(audio) / SAMPLE_RATE, 3)))
        if demo_info is not None:
            self.state["source"]["demo_song"] = demo_song
        source_label = demo_info["title"] if demo_info else kind
        analysis_label = "テンポ" if demo_info else "推定テンポ"
        display(HTML(
            f"<h3>音源の準備ができました</h3><p>{escape(source_label)} / {len(audio) / SAMPLE_RATE:.1f}秒<br>"
            f"{analysis_label}：{analysis['bpm']:.1f} BPM / キー：{escape(analysis['key'])} / "
            f"スケール：{escape(analysis['scale'])}<br>{escape(analysis.get('note', ''))}</p>"
        ))
        display(Audio(audio, rate=SAMPLE_RATE, normalize=False))
        print("次にセル4の設定を選び、左の▶を押してアレンジを生成してください。")
        return True

    def generate(self, *, auto_tempo=True, bpm=120, auto_key_scale=True, root="C", scale="major",
                 auto_beat=True, beat_offset=0, instrument="Electric Piano", add_chords=True,
                 add_drums=True, add_melody=True, original_gain=.7, chord_gain=.6,
                 drum_gain=.8, melody_gain=.6, humanise=0, seed=42):
        """フォームの値で生成し、聴き比べ用音声と保存ファイルを用意します。"""
        self._invalidate_result()
        if self.state["audio"] is None or self.state["analysis"] is None:
            print("先にセル3で音源を用意してください。")
            return False
        analysis = self.state["analysis"]
        config = ArrangementConfig(
            bpm=analysis["bpm"] if auto_tempo else bpm,
            root=analysis["key"] if auto_key_scale else root,
            scale=analysis["scale"] if auto_key_scale else scale,
            beat_offset=analysis["beat_offset"] if auto_beat else beat_offset,
            instrument=instrument, add_chords=add_chords, add_drums=add_drums,
            add_melody=add_melody, original_gain=original_gain, chord_gain=chord_gain,
            drum_gain=drum_gain, melody_gain=melody_gain, humanise=humanise, seed=seed,
        )
        try:
            melody = self.state["melody_notes"]
            if add_melody and melody_gain > 0 and melody is None:
                print("メロディの音の高さを推定しています。初回は時間がかかる場合があります。")
                melody = extract_melody(self.state["audio"], SAMPLE_RATE)
                self.state["melody_notes"] = melody
            print("伴奏・ドラムを生成し、保存ファイルを作っています。")
            result = arrange(self.state["audio"], SAMPLE_RATE, config,
                             melody_notes=melody if add_melody else [])
            artifacts = self._write_artifacts(result, config)
        except ValueError as error:
            print("生成できませんでした：" + str(error))
            return False
        except Exception:
            print("生成を完了できませんでした。セル1から順に実行し直し、短いデモ音源で試してください。")
            return False
        self.state.update(result=result, artifacts=artifacts, config=config)
        self._render_result(result)
        print("生成が完了しました。設定を変えた後はセル4を再実行し、セル5で結果を保存してください。")
        return True

    def _write_artifacts(self, result, config):
        target = Path(tempfile.mkdtemp(prefix="result_", dir=self.work_dir))
        paths = {name: target / name for name in
                 ("mixed.wav", "accompaniment.wav", "arrangement.mid", "settings.json")}
        sf.write(paths["mixed.wav"], result["mixed"], SAMPLE_RATE, subtype="PCM_16")
        sf.write(paths["accompaniment.wav"], result["accompaniment"], SAMPLE_RATE, subtype="PCM_16")
        result["midi"].write(str(paths["arrangement.mid"]))
        settings = dict(version=VERSION, source=self.state["source"], arrangement=asdict(config),
                        sample_rate=SAMPLE_RATE,
                        note="BPM・開始位置は追加伴奏の設定です。原曲の速度は変えません。")
        paths["settings.json"].write_text(json.dumps(settings, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        archive = target / "arrangement_results.zip"
        with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as zipped:
            for name, path in paths.items():
                zipped.write(path, arcname=name)
            zipped.writestr("README.txt", "AI作曲アレンジ\n\nmixed.wav：原曲と追加パートのミックス\naccompaniment.wav：追加パートだけ\narrangement.mid：追加パートの音符\nsettings.json：今回の設定\n\nMIDIには原曲の録音を含みません。ミックスの公開・共有は音源の利用条件を確認してください。\n")
        paths["arrangement_results.zip"] = archive
        return paths

    def _render_result(self, result):
        for title, audio in [("原曲（使用した区間）", self.state["audio"]),
                             ("追加パートだけ", result["accompaniment"]),
                             ("原曲と追加パートのミックス", result["mixed"])]:
            display(HTML(f"<h3>{title}</h3>"))
            display(Audio(audio, rate=SAMPLE_RATE, normalize=False))
        for warning in result.get("warnings", []):
            print(warning)
        self._plot_notes(result)

    @staticmethod
    def _plot_notes(result):
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(10, 2.8))
        for name, colour, label in [("chords", "#0f766e", "Chords"),
                                    ("melody", "#2563eb", "Extracted melody")]:
            for i, note in enumerate(result.get("notes", {}).get(name, [])):
                ax.plot([note.start, note.end], [note.pitch, note.pitch], color=colour,
                        linewidth=3, alpha=.7, label=label if i == 0 else None)
        ax.set(xlabel="Time (s)", ylabel="MIDI note", title="Added notes (drums omitted)")
        ax.grid(alpha=.2)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(loc="upper right")
        fig.tight_layout()
        display(HTML("<p>横軸は時間、縦軸は音の高さです。青は抽出メロディ、緑はコード伴奏です。</p>"))
        display(fig)
        plt.close(fig)

    def download(self, kind="ZIP一式"):
        """Colabではダウンロード、ローカルでは表示場所に依存しないリンクを作ります。"""
        filenames = {"ZIP一式": "arrangement_results.zip", "ミックスWAV": "mixed.wav",
                     "MIDI": "arrangement.mid", "追加パートWAV": "accompaniment.wav"}
        filename = filenames.get(kind)
        if filename is None:
            print("保存する形式をZIP一式・ミックスWAV・MIDI・追加パートWAVから選んでください。")
            return False
        path = self.state["artifacts"].get(filename)
        if path is None or not Path(path).is_file():
            print("保存できる結果がありません。先にセル4を実行して生成してください。")
            return False
        try:
            try:
                from google.colab import files
            except ImportError:
                encoded = base64.b64encode(Path(path).read_bytes()).decode("ascii")
                name = escape(filename, quote=True)
                display(HTML(f'<a download="{name}" href="data:application/octet-stream;base64,{encoded}">{name} をダウンロード</a>'))
            else:
                files.download(str(path))
        except Exception:
            print("ダウンロードを開始できませんでした。ブラウザのダウンロード設定を確認し、セル5を再実行してください。")
            return False
        return True
''',
}

for filename, source in module_sources.items():
    path = Path(filename)
    path.write_text(source, encoding="utf-8")
    # 同じセルを再実行したときも、今保存したコードを読みます。
    Path(importlib.util.cache_from_source(str(path))).unlink(missing_ok=True)

for filename in reversed(module_sources):
    sys.modules.pop(Path(filename).stem, None)
importlib.invalidate_caches()
for filename in module_sources:
    importlib.import_module(Path(filename).stem)

# 読み込み直したコードで、次のセルから新しく音源を扱います。
globals().pop("music_session", None)
print("プログラムの準備ができました。セル3で音源を選んでください。")


## 3. 音源を選んで読み込む

**YouTubeならURLを貼って ▶ を押します。** URLを空欄にすると、`demo_song` で選んだ曲を使います。
デモには「きらきら星」「歓喜の歌」もあります。まずは聴き慣れた曲で試してみてください。

PCのファイルを使うときは `input_source` を「PCの音源」に変えて実行します。
YouTube・PC音源は開始位置と長さを指定できます。たとえば開始30秒・長さ20秒なら、30〜50秒の区間を使います。

YouTubeから取れないときは、公式配布元のMP3を「PCの音源」で読み込めます。
フリー音源でも接続が制限される場合があります。

In [ ]:
#@title 3. 音源を選んで読み込む { display-mode: "form" }
#@markdown **YouTubeを使う：** 動画のURLを貼り、左の ▶ を押してください。
youtube_url = "" #@param {type:"string"}
#@markdown 通常は「URLがあればYouTube」のままでOKです。
input_source = "URLがあればYouTube" #@param ["URLがあればYouTube", "YouTube", "PCの音源", "デモ音源"]
#@markdown **デモで試す：** URLを空欄にして曲を選びます。
demo_song = "オリジナル（ポップ）" #@param ["オリジナル（ポップ）", "オリジナル（ゆったり）", "きらきら星（冒頭）", "歓喜の歌（冒頭）"]
#@markdown **使う区間：** YouTube・PC音源の開始位置と長さです。デモは曲ごとの長さで再生します。
start_seconds = 0 #@param {type:"number"}
duration_seconds = 20 #@param {type:"number"}

from notebook_session import NotebookSession
if "music_session" not in globals() or not isinstance(music_session, NotebookSession):
    music_session = NotebookSession()
source_ready = music_session.prepare_source(input_source, youtube_url=youtube_url,
                                            start=start_seconds, duration=duration_seconds,
                                            demo_song=demo_song)


## 4. アレンジを生成する

最初はそのまま ▶ を押してみましょう。原曲・追加パート・ミックスを聴き比べられます。
慣れたらドラムを外したり、楽器を変えたりしてみてください。

手動でテンポやキーを変えるときは、対応する `auto_…` のチェックを外します。
テンポが変わるのは追加する伴奏だけです。原曲の速さは変わりません。
メロディ抽出は音の高さを拾う処理なので、歌声だけを分離するものではありません。

In [ ]:
#@title 4. アレンジを生成する { display-mode: "form" }
#@markdown **テンポと拍の位置**：自分で調整したい項目だけ、自動設定のチェックを外します。
auto_tempo = True #@param {type:"boolean"}
bpm = 120 #@param {type:"number"}
auto_beat = True #@param {type:"boolean"}
beat_offset = 0.0 #@param {type:"number"}
#@markdown **キー・音階・音色**：長調と短調を聴き比べるときは auto_key_scale をオフにします。
auto_key_scale = True #@param {type:"boolean"}
root = "C" #@param ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
scale = "major" #@param ["major", "natural_minor", "harmonic_minor", "dorian", "mixolydian", "pentatonic"]
instrument = "Electric Piano" #@param ["Electric Piano", "Acoustic Grand Piano", "Acoustic Guitar", "Electric Guitar", "Strings", "Synth Lead", "Music Box"]
#@markdown **加えるパート**：必要な音だけをオンにしてください。
add_chords = True #@param {type:"boolean"}
add_drums = True #@param {type:"boolean"}
add_melody = True #@param {type:"boolean"}
#@markdown **音量と揺らぎ**：original_gain を0にすると、加えた音だけを聴けます。
original_gain = 0.7 #@param {type:"slider", min:0, max:1.5, step:0.05}
chord_gain = 0.6 #@param {type:"slider", min:0, max:1.5, step:0.05}
drum_gain = 0.8 #@param {type:"slider", min:0, max:1.5, step:0.05}
melody_gain = 0.6 #@param {type:"slider", min:0, max:1.5, step:0.05}
humanise = 0.0 #@param {type:"slider", min:0, max:1, step:0.1}
seed = 42 #@param {type:"integer"}

if "music_session" not in globals():
    print("先にセル3で音源を読み込んでください。")
else:
    music_session.generate(auto_tempo=auto_tempo, bpm=bpm, auto_beat=auto_beat,
                           beat_offset=beat_offset, auto_key_scale=auto_key_scale,
                           root=root, scale=scale, instrument=instrument,
                           add_chords=add_chords, add_drums=add_drums, add_melody=add_melody,
                           original_gain=original_gain, chord_gain=chord_gain,
                           drum_gain=drum_gain, melody_gain=melody_gain,
                           humanise=humanise, seed=seed)


## 5. 結果を保存する

形式を選び、▶ を押します。「ZIP一式」にはミックス・追加パートのWAV、MIDI、設定が入ります。
WAVは音声、MIDIは音符データです。MIDIには原曲の録音は含まれません。
ランタイムを切断する前に、残したい結果を自分のPCへ保存してください。

In [ ]:
#@title 5. 結果を保存する { display-mode: "form" }
save_format = "ZIP一式" #@param ["ZIP一式", "ミックスWAV", "MIDI", "追加パートWAV"]

if "music_session" not in globals():
    print("先にセル3で音源を読み込み、セル4で生成してください。")
else:
    music_session.download(save_format)


## こんな聴き比べもできます

- `auto_key_scale` をオフにし、Cのmajorとnatural_minorを比べます。
- `add_drums` をオフにし、リズムの印象がどう変わるかを聴きます。
- `original_gain` を0にし、追加されたパートだけを聴きます。
- `humanise` を少し上げ、演奏のタイミングや強弱を揺らしてみます。

変更したらセル4を実行して聴き比べます。気に入ったらセル5で保存しましょう。